In [573]:
library(readr)
library(dplyr)
library(janitor)
library(stringr)
library(tidyr)
library(stringdist)

In [574]:
# Raw scrape: every dashboard question + answer (the scraper does NO matching now).
scraped <- read_csv("data/brfss_export_data/brfss_2014_phr_1_8_11_all_questions.csv") |> clean_names()

# Canonical BRFSS variables: one row per "<Feature> - <Response>" in `Sheet`.
# This is the set of variables we match each scraped question/answer to.
brfss <- read_csv("data/brfss_all_categories.csv")

Rows: 540 Columns: 172
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (171): topic, question, total_yes_percent, total_no_percent, total_docto...
dbl   (1): area



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 391 Columns: 23
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (12): Sheet, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11
dbl  (8): PHR2_pct_diff, PHR3_pct_diff, PHR4_pct_diff, PHR5_pct_diff, PHR6_p...
lgl  (3): PHR1_pct_diff, PHR8_pct_diff, PHR11_pct_diff

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [575]:
# ── Text helpers ─────────────────────────────────────────────────────────────
clean <- function(x) {                 # human label -> snake_case key
  x <- gsub("[^A-Za-z0-9]+", "_", x)
  gsub("^_+|_+$", "", x)
}

# Filler tokens: words that appear in the dashboard's verbose answer wording but
# not in the terse canonical labels (articles, comparison words, "example", …).
# Dropping them lets "within the past year (anytime less than 12 months ago)"
# match the canonical "Within the past year".
FILLER <- c("the","a","an","to","or","but","less","than","anytime","ago",
            "kind","of","example","examples","only","and","in")

tokens <- function(x) {
  x <- tolower(gsub("\n", " ", x))
  x <- gsub("[^a-z0-9]+", " ", x)
  setdiff(strsplit(trimws(x), "\\s+")[[1]], FILLER)
}

# ── Canonical targets ────────────────────────────────────────────────────────
# `Sheet` is "<Feature> - <Response>". Split on the LAST " - " (greedy) so the
# feature and response can be matched separately: feature via the curated
# crosswalk, response automatically within that feature's small response set.
canon <- brfss %>%
  transmute(
    sheet    = Sheet,
    feature  = trimws(sub("^(.*) - .*$", "\\1", Sheet)),
    response = trimws(sub("^.* - (.*)$", "\\1", Sheet)),
    variable = tolower(clean(Sheet))
  )

In [576]:
# ── Reshape to one row per area × question × answer option ────────────────────
scraped_long <- scraped %>%
  pivot_longer(cols = starts_with("total_"),
               names_to = "answer", values_to = "percent") %>%
  filter(!is.na(percent)) %>%
  mutate(
    answer  = answer %>% str_remove("^total_") %>% str_remove("_percent$"),
    percent = parse_number(percent)
  )

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `percent = parse_number(percent)`.
Caused by warning:
! 67 parsing failures.
row col expected actual
 22  -- a number     --
 23  -- a number     --
 40  -- a number     --
 41  -- a number     --
 42  -- a number     --
... ... ........ ......
See problems(...) for more details.”


In [577]:
# scraped_long

In [578]:
# ── CURATED question → canonical-feature crosswalk ───────────────────────────
# The ONE place feature matching is decided. Each dashboard question maps to
# exactly one canonical BRFSS feature (the short label used in brfss `Sheet`).
# When the "unmatched questions" report (further down) flags a question that DOES
# have a canonical feature, add a line here. Questions absent from this list have
# no canonical equivalent (e.g. "Activity on the Internet: …", PSA / shot /
# walking-for-transportation items) and are intentionally dropped.
# NOTE: two scraped questions can map to two DIFFERENT canon features even when
# they look similar (Current Asthma vs Current Asthma 2; Diabetes vs Diabetes
# Status) — they are different measures, so keep them distinct to avoid collisions.
question_to_feature <- c(
  "A1C Test the Past Year"                                                                = "A One C",
  "Any Cancer"                                                                            = "Any Cancer",
  "Any Drinking in the Past Month"                                                        = "Any Drinking Past Month",
  "Arthritis"                                                                             = "Arthritis",
  "Asthma Status"                                                                         = "Asthma Status",
  "Binge Drinking in the Past Month"                                                      = "Binge Drinking",
  "Binge Drinking or Heavy Alcohol Consumption, Females Age 18-44"                        = "Binge-Heavy Drinking - Femal",
  "Blind or Have Serious Difficulty Seeing"                                               = "Blind",
  "Body Mass Index (BMI) 3 Categories"                                                    = "BMI 3 Categories",
  "Body Mass Index (BMI) 4 Categories"                                                    = "BMI 4 Categories",
  "Body Mass Index (BMI) 5 Categories"                                                    = "BMI 5 Categories",
  "Most Recent Exam Sigmoidoscopy or Colonoscopy"                                         = "ClnscpySgmscpy",
  "Colonoscopy in the Past 10 Years, Age 50-75"                                           = "Col Past 10 Yrs 50-75",
  "Chronic Obstructive Pulmonary Disease (COPD)"                                          = "COPD",
  "Coronary Heart Disease"                                                                = "Coronary Heart Disease",
  "Current Asthma"                                                                        = "Current Asthma",
  "Current Asthma Status"                                                                 = "Current Asthma 2",
  "Current Smoker"                                                                        = "Current Smoker",
  "Cardiovascular Disease"                                                                = "CVD",
  "Dentist Visit in the Past Year"                                                        = "Dentist Visit Past Year",
  "Depressive Disorders"                                                                  = "Depressive Disorder",
  "Diabetes"                                                                              = "Diabetes",
  "Age When Diagnosed With Diabetes"                                                      = "Diabetes Age",
  "Ever Attended Diabetes Education"                                                      = "Diabetes Education",
  "Time Since Last Eye Exam"                                                              = "Diabetes Eye Exam",
  "Diabetes Status"                                                                       = "Diabetes Status",
  "Tested for Diabetes or High Blood Sugar"                                               = "Diabetes Test",
  "Have Serious Difficulty Concentrating, Remembering, or Making Decisions"               = "Difficulty Concentrating",
  "Have Difficulty Doing Errands Alone"                                                   = "Difficulty Doing Errands Alo",
  "Have Serious Difficulty Dressing or Bathing"                                           = "Difficulty Dressing",
  "Have Serious Difficulty Walking or Climbing Stairs"                                    = "Difficulty Walking",
  "Disability Status"                                                                     = "Disability Status",
  "Time Since Last Sigmoidoscopy or Colonoscopy"                                          = "DurationClnscpySgmy",
  "Time Since Blood Stool Test"                                                           = "DurationHomeBloodStoolTest",
  "Ever Had Blood Stool Test"                                                             = "Ever Had Blood Stool Test",
  "Ever Had HIV Test"                                                                     = "Ever Had HIV Test",
  "Ever Had Sigmoidoscopy or Colonoscopy"                                                 = "Ever Had Sgmscpy-clnscpy",
  "Ever Smoked"                                                                           = "Ever Smoked",
  "General Health Fair to Poor"                                                           = "Fair or Poor Health",
  "Flu Shot in the Past Year"                                                             = "Flu Shot in Past Year",
  "Flu Shot in the Past Year, Age 18-64"                                                  = "Flu Shot, 18-64",
  "Flu Shot in the Past Year, Age 65+"                                                    = "Flu Shot, 65+",
  "Location of Flu Shot"                                                                  = "Flu Vaccine Location",
  "Blood Stool Test in the Past Year, Age 50-75"                                          = "FOBT Past Yr",
  "Frequency of Smoking"                                                                  = "Frequency of Smoking",
  "General Health"                                                                        = "General Health",
  "Have At Least One Personal Doctor"                                                     = "Have At Least One Personal D",
  "Health Care Coverage"                                                                  = "Health Care Coverage",
  "Health Care Coverage, Age 18-64"                                                       = "Health Care Coverage Age 18-",
  "Have Personal Doctor"                                                                  = "Health Care Provider",
  "Heart Attack"                                                                          = "Heart Attack",
  "Heart Disease"                                                                         = "Heart Disease",
  "Heavy Alcohol Consumption"                                                             = "Heavy Drinking",
  "Heavy Alcohol Consumption, Females"                                                    = "Heavy Drinking - Females",
  "Heavy Alcohol Consumption, Males"                                                      = "Heavy Drinking - Males",
  "Days Poor Health Interfered With Usual Activities - 14+ Days"                          = "Hlth Affected Activ 14+ Days",
  "Days Poor Health Interfered With Usual Activities - 5+ Days"                           = "Hlth Affected Activ 5+ Days",
  "Had Hysterectomy"                                                                      = "Hysterectomy",
  "Taking Insulin"                                                                        = "Insulin",
  "Kidney Disease"                                                                        = "Kidney Disease",
  "Time Since Last Dentist Visit"                                                         = "Last Dentist Visit",
  "Time Since Quit Smoking"                                                               = "Last Smoked",
  "Leisure Time Physical Activity"                                                        = "Leisure Time PA",
  "Lifetime Asthma"                                                                       = "Lifetime Asthma",
  "Ever Had a Mammogram"                                                                  = "Mammogram",
  "Mammogram in the Past 2 Years, Age 40+"                                                = "Mammogram Past 2 Yrs 40+",
  "Mammogram in the Past 2 Years, Age 50+"                                                = "Mammogram Past 2 Yrs 50+",
  "Mammogram in the Past 2 Years, Age 50-74"                                              = "Mammogram Past 2 Yrs 50-74",
  "Medical Cost"                                                                          = "Medical Cost",
  "Obese"                                                                                 = "Obese",
  "Other Cancer"                                                                          = "Other Cancer",
  "Overweight or Obese"                                                                   = "Overweight or Obese",
  "Pap Smear Test in the Past 3 Years, Age 21-65"                                         = "Pap Test Past 3 Yrs 21-65",
  "Ever Had Pneumonia Shot"                                                               = "Pneumonia Shot, 18+",
  "Ever Had Pneumonia Shot, Age 65+"                                                      = "Pneumonia Shot, 65+",
  "Days of Poor Mental Health - 14+ Days"                                                 = "Poor Mental Health 14+ Days",
  "Days of Poor Mental Health - 5+ Days"                                                  = "Poor Mental Health 5+ Days",
  "Days of Poor Physical Health - 14+ Days"                                               = "Poor Physical Health 14+ Day",
  "Days of Poor Physical Health - 5+ Days"                                                = "Poor Physical Health 5+ Days",
  "Prediabetes"                                                                           = "Pre-Diabetes",
  "Quit Smoking in the Past Year"                                                         = "Quit Smoking in Past Yr",
  "Number of Permanent Teeth Removed"                                                     = "RemoveTeeth",
  "Time Since Routine Checkup"                                                            = "Routine Checkup",
  "Routine Checkup in the Past Year"                                                      = "Routine Checkup in the Past",
  "Served On Active Duty in the United States Armed Forces"                               = "Served on Active Duty",
  "Sigmoidoscopy in the Past 5 Years and Blood Stool Test in the Past 3 Years, Age 50-75" = "Sigm and Blood Stool Age 50-",
  "Sigmoidoscopy in the Past 5 Years, Age 50-75"                                          = "Sigm Past 5 Yrs Age 50-75",
  "Skin Cancer"                                                                           = "Skin Cancer",
  "Smokeless Tobacco Use"                                                                 = "Smokeless Tobacco Use",
  "Smoker Status"                                                                         = "Smoker Status",
  "Stroke"                                                                                = "Stroke",
  "Up-To-Date On Colorectal Cancer Screening, Age 50-75"                                  = "Up-To-Date CRC Scrn Age 50-7"
)

scraped_long <- scraped_long %>%
  mutate(feature = unname(question_to_feature[question]))

In [579]:
# ── Automatic answer → canonical-response matching (within the matched feature)─
# The dashboard writes each bucket as "<canonical label> (clarification…)", e.g.
# "within the past 3 years (2 years but less than 3 years ago)". The canonical
# label is therefore a leading PREFIX of the answer's tokens, so we match the
# longest canonical response whose tokens are a contiguous prefix of the answer's
# tokens. Prefix (not mere containment) is what prevents the secondary boundary
# number — the "2" in "…3 years (2 years…)" — from wrongly matching "2 years".
# The synonym map handles the few responses that aren't a clean prefix
# (reordered/abbreviated wordings, non-lexical categories). A synonym only fires
# when its target is an actual response of the matched feature, so it can't leak
# across features.
response_synonyms <- c(
  "normal"                                                                   = "Recommended Range",          # BMI 3/4/5 Categories
  "neither_overweight_or_obese"                                              = "Recommended Range",          # BMI 3 Categories
  "never_smoker"                                                             = "Never smoked",               # Smoker Status
  "41"                                                                       = "40 years or younger",        # Diabetes Age (youngest)
  "65"                                                                       = "65 years or older",          # Diabetes Age (oldest)
  "yes_but_female_told_only_during_pregnancy"                                = "Yes, but during pregnancy",  # Diabetes
  "no_pre_diabetes_or_borderline_diabetes"                                   = "No, borderline\nor pre-diabetes",  # Diabetes
  "a_doctor_s_office_or_health_maintenance_organization_hmo"                 = "Doctor's office\nHMO",       # Flu Vaccine Location
  "another_type_of_clinic_or_health_center_example_a_community_health_center" = "Clinic\nHealth center",       # Flu Vaccine Location
  "a_senior_recreation_or_community_center"                                  = "Senior\nRecreation\nCommunity center",  # Flu Vaccine Location
  "received_vaccination_in_canada_mexico"                                    = "Canada\nMexico",             # Flu Vaccine Location
  "some_other_kind_of_place"                                                 = "Other"                       # Flu Vaccine Location
)

match_response <- function(answer, feature) {
  cands <- canon$response[canon$feature == feature]
  if (length(cands) == 0) return(NA_character_)
  syn <- unname(response_synonyms[answer])          # NA if answer not a synonym
  if (!is.na(syn) && syn %in% cands) return(syn)
  at <- tokens(answer)
  best <- NA_character_; best_len <- 0
  for (r in cands) {
    rt <- tokens(r)
    # canonical tokens must be a contiguous prefix of the answer's tokens;
    # prefer the longest such response (most specific).
    if (length(rt) > best_len &&
        length(rt) <= length(at) &&
        all(rt == at[seq_along(rt)])) {
      best_len <- length(rt); best <- r
    }
  }
  best
}

scraped_long <- scraped_long %>%
  rowwise() %>%
  mutate(response = if (is.na(feature)) NA_character_ else match_response(answer, feature)) %>%
  ungroup() %>%
  mutate(
    sheet    = if_else(!is.na(feature) & !is.na(response),
                       paste(feature, response, sep = " - "), NA_character_),
    variable = if_else(!is.na(sheet), tolower(clean(sheet)), NA_character_)
  )

In [580]:
# ── Review reports — the only things needing human attention ──────────────────
feat_list <- unique(canon$feature)
suggest_feature <- function(q)
  feat_list[which.min(stringdist(tolower(q), tolower(feat_list), method = "jw", p = 0.1))]

# (a) Questions with NO crosswalk entry. Most genuinely have no canonical
#     equivalent and are dropped; if one SHOULD map, add it to
#     `question_to_feature` above. `suggested_feature` is the closest canonical
#     feature (Jaro-Winkler) — verify it before trusting.
unmatched_questions <- scraped_long %>%
  filter(is.na(feature)) %>% distinct(question) %>%
  rowwise() %>% mutate(suggested_feature = suggest_feature(question)) %>% ungroup()
cat("Questions with NO canonical feature:", nrow(unmatched_questions), "\n")
print(unmatched_questions, n = Inf)

# (b) Question mapped to a feature, but the answer matched no canonical response
#     (coarser canon granularity, no equivalent bucket, etc.). Decide per case:
#     add a response synonym, fix the crosswalk, or accept the drop.
unmatched_responses <- scraped_long %>%
  filter(!is.na(feature), is.na(response)) %>% distinct(feature, answer)
cat("\nAnswers under a matched feature with NO canonical response:",
    nrow(unmatched_responses), "\n")
print(unmatched_responses, n = Inf)

Questions with NO canonical feature: 88 
# A tibble: 88 × 2
   question                                                    suggested_feature
   <chr>                                                       <chr>            
 1 Ever Had Hepatitis B Shot                                   Ever Had HIV Test
 2 Ever Had HPV Shot                                           Ever Had HIV Test
 3 Ever Had Meningitis Shot                                    Ever Had HIV Test
 4 Had All Hepatitis B Shots                                   Have At Least On…
 5 Had All HPV Shots                                           Have At Least On…
 6 Had Shingles Shot, Age 50+                                  Sigm and Blood S…
 7 Number of Hepatitis B Shots                                 Routine Checkup …
 8 Number of HPV Shots                                         Ever Had Blood S…
 9 Received Tetanus Shot Since 2005                            Routine Checkup …
10 Advised To Reduce or Quit Drinking            

In [581]:
# ── Final matched output ─────────────────────────────────────────────────────
# Keep only fully-matched rows (question -> feature AND answer -> response).
matched <- scraped_long %>%
  filter(!is.na(variable)) %>%
  transmute(variable, sheet, feature, response, answer, phr = area, percent) %>%
  arrange(variable, phr)

cat("Matched", nrow(matched), "rows ->", n_distinct(matched$variable),
    "canonical variables across", n_distinct(matched$phr), "PHRs.\n")

# Safety: every variable we emit must exist in the canonical list.
stopifnot(all(matched$variable %in% canon$variable))

# Safety: each (variable, phr) must be unique. A duplicate means two questions
# or two answers collapsed onto the same canonical variable — inspect `clash`.
clash <- matched %>% count(variable, phr) %>% filter(n > 1)
if (nrow(clash) > 0) {
  cat("\nWARNING: duplicate (variable, phr) rows:\n"); print(as.data.frame(clash))
}
stopifnot(nrow(clash) == 0)

write_csv(matched %>% select(variable, phr, percent),
          "data/created/brfss_scraped_2014.csv")

Matched 693 rows -> 231 canonical variables across 3 PHRs.


In [586]:
matched[rowSums(is.na(matched)) > 0, ]

unique(brfss_features)

variable,sheet,feature,response,answer,phr,percent
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
a_one_c_no,A One C - No,A One C,No,no,1,NA
a_one_c_yes,A One C - Yes,A One C,Yes,yes,1,NA
diabetes_age_40_years_or_younger,Diabetes Age - 40 years or younger,Diabetes Age,40 years or younger,41,1,NA
diabetes_age_41_to_64,Diabetes Age - 41 to 64,Diabetes Age,41 to 64,41_64,1,NA
diabetes_age_65_years_or_older,Diabetes Age - 65 years or older,Diabetes Age,65 years or older,65,1,NA
diabetes_education_no,Diabetes Education - No,Diabetes Education,No,no,1,NA
diabetes_education_yes,Diabetes Education - Yes,Diabetes Education,Yes,yes,1,NA
diabetes_eye_exam_2_or_more_years,Diabetes Eye Exam - 2 or more years,Diabetes Eye Exam,2 or more years,2_or_more_years_ago,1,NA
diabetes_eye_exam_never,Diabetes Eye Exam - Never,Diabetes Eye Exam,Never,never,1,NA


Sheet
<chr>
A One C - No
A One C - Yes
Advised to Change Eating Hab - No
Advised to Change Eating Hab - Yes
Advised to Cut Down Salt - Do not use salt
Advised to Cut Down Salt - No
Advised to Cut Down Salt - Yes
Advised to Exercise - No
Advised to Exercise - Yes
